# 07 · Phase A (GPU encoder): query embedding, search, retriever grid and rerankers

Plan sections 4 and 5 (phase A). Everything that needs **no LLM** runs here, with the encoder on the GPU:

| step | what | time on the GPU encoder |
|---|---|---|
| query embeddings | the 100 questions (dense, sparse, ColBERT) | seconds |
| search | each signal's top-50 per index → `data/cache/search.jsonl` | minutes |
| grid | L1, S1, D1, C1, H1, H2, M1, F0, T1, T0, T2, T3, T2+T3, gate off and on | minutes |
| rerankers | X1 (bge-reranker-v2-m3) and X2 (jina-reranker-v3), gate off and on | **~1 h** (≈ 9 h on the CPU) |

The methods that need Qwen (Q1, Q2, F1 and the agent T4) run in **notebook 08**, after the swap to vLLM.

> **⚠ STOP vLLM BEFORE RUNNING THIS NOTEBOOK.** The GPU encoder and vLLM do not fit on the card together.
> ```bash
> docker compose stop vllm encoder                   # vLLM off; the CPU encoder shares port 8081
> docker compose up -d qdrant && docker compose up -d --force-recreate encoder-gpu            # phase A
> ```
> The guard cell below refuses to run while vLLM answers on port 8000, and the reranker cell refuses to run on a CPU encoder unless you set `ALLOW_CPU_RERANK = True`.

Measured on the RTX 3090 (fp32, TF32 off), per question with 50 candidates: bge 5.6–7.2 s on base units and ~2.4 s on windows; jina ~16 s and ~8 s. The CPU took 33 s (bge) and 141 s (jina) for the same base-unit call, and its scores agree with the GPU's to within 4e-5, with identical rankings.

Everything is cached and every run resumes from `partial.jsonl`; runs whose `hits.parquet` exists are skipped. For `v2`, set `PARAMS_VERSION = "v2"` and run this notebook and then notebook 08 again.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
import json, time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from csrag import runs
from csrag.corpus import COLLECTIONS, Corpus
from csrag.retrievers import Pipeline, Resources, check_rrf, grid, method_config, prompt_hashes
from csrag.retrievers.base import get_params
from csrag.retrievers.query_llm import vllm_healthy

PARAMS_VERSION = "v1"
ALLOW_CPU_RERANK = False  # True only if you accept ~9 h of CPU reranking

assert not vllm_healthy(), "vLLM is running: `docker compose stop vllm`, then `docker compose up -d qdrant && docker compose up -d --force-recreate encoder-gpu`"
corpus = Corpus(DATA)
res = Resources.connect(corpus, DATA, with_llm=False)
health = res.encoder.wait_ready()
print("encoder on", health["device"], health.get("gpu") or "", "| dtype", health["dtype"])
if health["device"] != "cuda":
    print("WARNING: the encoder runs on the CPU. Phase A is meant for `encoder-gpu`.")
for index in corpus.indexes():
    n = res.client.get_collection(COLLECTIONS[index]).points_count
    assert n == len(corpus.units(index)), f"{index}: {n} points, run notebook 05"
    assert not res.colbert(index).missing, f"{index}: ColBERT vectors missing, run notebook 04"
print("Qdrant collections and ColBERT indexes ready")

In [ ]:
questions = [json.loads(l) for l in open(DATA / "eval" / "questions.jsonl", encoding="utf-8")]
texts = [q["question"] for q in questions]
labels = {q["question_id"]: q["engine_type"] for q in questions}  # passed to F0 (the oracle) only
prompts = prompt_hashes()
configs = grid(PARAMS_VERSION, prompts=prompts)
LLM_METHODS = ("Q1", "Q2", "F1", "T4")  # notebook 08
phase_a = [c for c in configs if c["method"] not in LLM_METHODS]
pd.Series([c["method"] for c in phase_a]).value_counts().sort_index().rename("phase A runs").to_frame().T

## Query embeddings

The 100 questions in dense, sparse and ColBERT form (`data/embeddings/queries_questions.npz`, shared by both indexes). BM25 query vectors are computed on the fly (weight 1 per stem). HyDE passages and agent queries are embedded in notebook 08 by the CPU encoder; the two devices agree to within ~1e-6 on bge-m3 dense vectors.

In [ ]:
t0 = time.time()
q_emb = res.qemb["questions"].get(texts, res.encoder)
print(len(q_emb), "question embeddings in", round(time.time() - t0, 1), "s")

## Search

Each signal's raw top-50 for every question and index, with scores and latency, into `data/cache/search.jsonl`. `dense_exact` repeats the dense search with `exact=True` (the ANN check). Filtered (F0, F1), HyDE (Q1) and keyword (Q2) searches fill the same cache during the runs.

In [ ]:
SIGNALS = ["bm25", "bge_sparse", "dense", "dense_exact", "colbert"]
for index in corpus.indexes():
    p = Pipeline(method_config("D1", index, False, PARAMS_VERSION), res)
    for t in tqdm(texts, desc=index):
        q = p._query(t)
        for sig in SIGNALS:
            p.searcher.search(index, sig, q, None, get_params(PARAMS_VERSION)["candidates"])
print(len(res.caches.search), "cached search lists")

### Checks: ANN vs exact, and client-side RRF vs Qdrant's RRF (plan 8)

In [ ]:
rows = []
for index in corpus.indexes():
    p = Pipeline(method_config("H1", index, False, PARAMS_VERSION), res)
    ann_same, rrf_ok = 0, 0
    for t in texts:
        q = p._query(t)
        a = p.searcher.search(index, "dense", q, None, 50)["hits"]
        b = p.searcher.search(index, "dense_exact", q, None, 50)["hits"]
        ann_same += [u for u, _ in a] == [u for u, _ in b]
        r = check_rrf(p, t)
        rrf_ok += r["same_set"] and r["same_scores"] and r["same_order_ignoring_ties"]
    rows.append({"index": index, "questions": len(texts), "dense HNSW == exact": ann_same, "RRF == Qdrant RRF (k=61)": rrf_ok})
pd.DataFrame(rows)

## The retriever grid without LLM and rerankers (fast loop)

L1, S1, D1, C1, H1, H2, M1, F0, T1 and the fixed expansion rules T0, T2, T3, T2+T3 (4k / 8k / 16k), with the gate off and on, from the cached lists.

In [ ]:
def execute(cfgs, desc):
    ids = []
    for cfg in tqdm(cfgs, desc=desc):
        p = Pipeline(cfg, res, labels=labels if cfg["method"] == "F0" else None)
        ids.append(runs.execute(cfg, p.run, questions, res.meta(cfg["index"])))
    return ids

fast = [c for c in phase_a if c["method"] not in ("X1", "X2")]
t0 = time.time()
fast_ids = execute(fast, "grid")
print(len(fast_ids), "runs in", round(time.time() - t0), "s")

### Gate checks (plan 8)

* gate-off lists equal the raw top-50 (here: the single-signal runs' top-20 equal the cached raw list's first 20);
* gate-on lists keep between `min_keep` and `max_keep` hits (or the whole list when it has fewer than 2 × `min_keep`), and are always a prefix of the gate-off list.

In [ ]:
ga = get_params(PARAMS_VERSION)["gate_a"]
problems = []
for m, sig in {"L1": "bm25", "S1": "bge_sparse", "D1": "dense", "C1": "colbert"}.items():
    for index in corpus.indexes():
        rid_off = runs.run_id_of(method_config(m, index, False, PARAMS_VERSION, prompts=prompts))
        rid_on = runs.run_id_of(method_config(m, index, True, PARAMS_VERSION, prompts=prompts))
        off, on = runs.load_hits(rid_off), runs.load_hits(rid_on)
        st = {s["question_id"]: s for s in runs.load_stats(rid_on)}
        p = Pipeline(method_config(m, index, False, PARAMS_VERSION), res)
        for q in questions:
            qid = q["question_id"]
            a = list(off[off.question_id == qid].sort_values("rank")["unit_id"])
            b = list(on[on.question_id == qid].sort_values("rank")["unit_id"])
            raw = [u for u, _ in p.searcher.search(index, sig, p._query(q["question"]), None, 50)["hits"]]
            g = st[qid]["gates"][0]
            if a != raw[:20]:
                problems.append((m, index, qid, "gate-off != raw top-20"))
            if b != a[: len(b)]:
                problems.append((m, index, qid, "gate-on is not a prefix"))
            if g["n_in"] >= 2 * ga.min_keep and not (ga.min_keep <= g["n_kept"] <= ga.max_keep):
                problems.append((m, index, qid, f"kept {g['n_kept']}"))
print(len(problems), "problems", problems[:10])

## Rerankers X1 and X2 (GPU encoder, ~1 h)

> **⚠ vLLM must be stopped** (checked again below). The rerankers run here, on the GPU encoder, never during the vLLM phase.

Both rerankers get H1's fused list (at most 50, shorter when gate A is on) in the same order and return at most 20. First, the truncation check: the longest base unit plus the longest question goes through both models without truncation (bge `max_length` 8192, jina `max_doc_length` 8192).

In [ ]:
assert not vllm_healthy(), "vLLM is running: `docker compose stop vllm` before reranking"
device = res.encoder.health()["device"]
assert device == "cuda" or ALLOW_CPU_RERANK, (
    "the encoder runs on the CPU (~9 h of reranking): `docker compose stop encoder && docker compose up -d encoder-gpu`, "
    "or set ALLOW_CPU_RERANK = True")
longest = max(corpus.units("units").values(), key=lambda u: corpus.n_tokens(u["text"]))
q_long = max(texts, key=len)
tb = res.encoder.tokens("bge", q_long, [longest["text"]])
tj = res.encoder.tokens("jina", q_long, [longest["text"]])
print(longest["unit_id"], "| bge pair tokens", tb["pairs"][0], "/", tb["max_length"], "| jina doc tokens", tj["documents"][0], "/", tj["max_doc_length"])
assert tb["pairs"][0] <= tb["max_length"] and tj["documents"][0] <= tj["max_doc_length"]

In [ ]:
rerank_cfgs = [c for c in phase_a if c["method"] in ("X1", "X2")]
t0 = time.time()
rerank_ids = execute(rerank_cfgs, f"rerankers on {device}")
print(len(rerank_ids), "reranker runs in", round((time.time() - t0) / 60), "min")

In [ ]:
done = [r for r in runs.list_runs(params_version=PARAMS_VERSION) if runs.load_config(r)["retriever"] not in LLM_METHODS]
print(f"phase A complete: {len(done)} of {len(phase_a)} runs")

## Next: swap to phase B (vLLM)

```bash
docker compose stop encoder-gpu                    # frees the GPU
docker compose up -d qdrant vllm && docker compose up -d --force-recreate encoder           # CPU encoder + vLLM (VLLM_CONFIG=serve.retrieval.yaml in .env)
docker compose logs -f vllm                        # ~2.5 min to healthy
```
Then run notebook 08.